# Config

In [1]:
!pip install optuna==4.5.0
!pip install plotly==6.3.0

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [optuna]2m1/2 [optuna]
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 14.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]


In [19]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [3]:
import pandas as pd
import os
import json

# 1) SPECTER OPTUNA

### Preprocess

In [4]:
#del gen_dataset
from utils.dataset import gen_dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)


### Optuna

In [ ]:
from transformers import logging
import warnings
import optuna
from pipelines.fine_tune_models import optuna_objective_cv, convert_numpy_to_native

warnings.filterwarnings(
    "ignore",
    message="TypedStorage is deprecated"
)
# Desactiva solo los warnings
logging.set_verbosity_error()

#Definir modelo para realizar fine tuning
model_name = "allenai/specter"

# ----------- Lanzar la optimización -----------
results_dir = "/tmp/results/finetune/SPECTER"
os.makedirs(results_dir, exist_ok=True)
#Crear estudio de optuna
study = optuna.create_study(
    direction="maximize",
    #study_name="specter",
    #storage= f"sqlite:///{results_dir}/optuna_3.db",
    #load_if_exists=True  # evita sobreescribir si ya existe
)
X_train=np.array(X_train)

#Crear función objetivo
opt_model = optuna_objective_cv(X_train, y_train, df_decode=df_decode, n_classes=2, model_name = model_name,
                                sample_weights_loss=True, Test_mode=True)
#Optimización
n_trials=1
study.optimize(opt_model.objective, n_trials=n_trials)

# ----------- Mostrar mejores resultados -----------
print("Mejor f1-score:", study.best_value)
print("Mejores hiperparámetros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

#Read results from the best model
results_best_model = opt_model.get_results()

# Aplica la conversión
metrics_native = convert_numpy_to_native(results_best_model['metrics'])

# Imprime con formato limpio
print({'metrics': metrics_native})

In [ ]:
import optuna
from optuna.visualization import plot_param_importances, plot_contour
import matplotlib.pyplot as plt

# ---------- 1. Importancia de Hiperparámetros ----------
fig1 = plot_param_importances(study)
fig1.show()

# ---------- 2. Gráfico de Contorno 2D ----------
# Encuentra los 2 hiperparámetros más importantes
importances = optuna.importance.get_param_importances(study)
top_params = list(importances.keys())[:2]

plot_contour(study, params=["lr", "n_unfreeze"])
    

### Retrain and eval model with the best params

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pipelines.fine_tune_models import Pytorch_Pipeline, mlflow_ckeckpoint
from utils.dataset import CvCustom, TextDataset
from torch.utils.data import DataLoader

#Definir variables
model_name = "allenai/specter"
cv_function=CvCustom(df_decode)
X_train=np.array(X_train)

#Sin optuna
params={
    "lr": 3.452088271232921e-05,
    "batch_size":5,
    "n_unfreeze":12 #12 max
    }

extra_parms={
    "n_trials":n_trials
}
#Reentrenar con mejores hyperparámetros definidos por optuna
#params=study.best_params

#Train
results = []
for nfold, (train_idx, test_idx) in enumerate(cv_function.split(X_train)):
    #Split data
    xt, yt = X_train[train_idx], y_train[train_idx]
    xv, yv = X_train[test_idx], y_train[test_idx]
    #define tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    #Datasets
    train_ds = TextDataset(list(xt), yt, tokenizer)
    val_ds   = TextDataset(list(xv), yv, tokenizer)
    test_ds = TextDataset(list(X_test), y_test, tokenizer)
    #Loaders
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)
    # ---------- Modelo (capa de clasificación encima de SPECTER) ----------
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    pipeline = Pytorch_Pipeline(model_class=model, use_scheduler=None, max_epochs=200)
    #Train
    pipeline.set_params(**params)
    pipeline.fit_early_stopping(train_loader, val_loader, yt)
    #Get test results
    metrics = pipeline.eval_test(pipeline.best_model_state, test_loader)
    #Save results
    print(metrics)
    results.append(metrics["f1_score"])
    #MLflow
    pipeline.update_to_best_model() #The principal model will be the best model on validation set
    exp_info={
        "exp_name": "SPECTER_finetuning",
        "run_name":f"fold{nfold}"
    }
    #mlflow_ckeckpoint(exp_info, pipeline, extra_parms, test_loader, y_test, df_test, mode="server")


mean=np.mean(results)
std=np.std(results)
print("mean:", mean)
print("std:", std)

# 2) RoBERTa

### Preprocess

In [16]:
#del gen_dataset
from utils.dataset import gen_dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

### Optuna

#### Functions

In [9]:
#Pytorch
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import inspect
from torch.optim import AdamW
from transformers import get_scheduler
import gc
#Optuna
from torch.utils.data import DataLoader
import optuna
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from utils.dataset import CvCustom, TextDataset
#mlflow
import mlflow
import git
import os

#Pytorch pipeline
def get_sample_weights_loss(y):
  y = np.asarray(y, dtype=np.int64)
  class_counts = np.bincount(y)
  class_weights = 1.0 / class_counts
  class_weights = class_weights / class_weights.sum()

  return class_weights

def unfreeze_last_layers(model, n_unfreeze: int):
    """
    Descongela las últimas `n_unfreeze` capas de un modelo Hugging Face.
    Compatible con BERT, RoBERTa, DistilBERT, ALBERT, XLM-R, etc.

    Args:
        model (torch.nn.Module): Modelo Hugging Face (posiblemente envuelto en DataParallel).
        n_unfreeze (int): Número de capas a descongelar.

    Returns:
        None. Modifica el modelo en su lugar.
    """

    # Si el modelo está envuelto en DataParallel, acceder al .module
    model_to_unfreeze = model.module if isinstance(model, torch.nn.DataParallel) else model

    # Detectar backbone automáticamente
    backbone = None
    for attr in ["bert", "roberta", "distilbert", "albert", "xlm_roberta"]:
        if hasattr(model_to_unfreeze, attr):
            backbone = getattr(model_to_unfreeze, attr)
            break

    if backbone is None:
        raise AttributeError("❌ No se encontró un backbone conocido (bert/roberta/distilbert/albert/xlm_roberta).")
    
    # Obtener capas del encoder
    if hasattr(backbone.encoder, "layer"):
        encoder_layers = backbone.encoder.layer
    elif hasattr(backbone, "transformer") and hasattr(backbone.transformer, "layer"):
        encoder_layers = backbone.transformer.layer  # DistilBERT
    else:
        raise AttributeError("❌ No se encontró el atributo 'layer' en el encoder del backbone.")

    # Descongelar últimas n capas
    for layer in encoder_layers[-n_unfreeze:]:
        for p in layer.parameters():
            p.requires_grad = True

class Pytorch_Pipeline():
    def __init__(self, model_class, sample_weights_loss=None, max_epochs = 200, use_scheduler=None):
        #Set device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        #Modelo
        self.model_class = model_class
        self.model = None
        #Elementos del entrenamiento
        self.params = None
        self.sample_weights_loss = sample_weights_loss
        self.criterion = None
        self.optimizer = None
        self.batch_size = None
        self.scheduler=None
        self.max_epochs = max_epochs
        #scheduler
        self.use_scheduler=use_scheduler
        #Best model
        self.best_model_state=None

    def partial_fit(self, loader):
        self.model.to(self.device)
        self.model.train()
        
        for batch in loader:
            batch = {k: v.to(self.device) for k, v in batch.items()}
            self.optimizer.zero_grad()
            out = self.model(**{k: v for k, v in batch.items() if k != "labels"})
            logits = out.logits
            loss = self.criterion(logits, batch["labels"].to(self.device))
            loss.backward()
            self.optimizer.step()
            if self.use_scheduler is not None:
                self.scheduler.step()

        return self

    def predict(self, loader):
        self.model.eval()
        all_preds = []

        with torch.no_grad():
            for batch in loader:
                # mover batch al device
                batch = {k: v.to(self.device) for k, v in batch.items()}
                xb = {k: v for k, v in batch.items() if k != "labels"}
                yb = batch["labels"]

                outputs = self.model(**xb)
                logits = outputs.logits

                # predicciones
                preds = logits.argmax(dim=1)
                all_preds.append(preds.cpu())

        y_pred = torch.cat(all_preds).numpy()
        return y_pred

    def predict_and_evaluate(self, loader):
            self.model.eval()
            total_loss = 0.0
            total_samples = 0
            all_preds, all_targets = [], []

            with torch.no_grad():
                for batch in loader:
                    # mover batch al device
                    batch = {k: v.to(self.device) for k, v in batch.items()}
                    xb = {k: v for k, v in batch.items() if k != "labels"}
                    yb = batch["labels"]

                    outputs = self.model(**xb)
                    logits = outputs.logits

                    # calcular pérdida (soporta reduction='mean' o 'none')
                    loss_val = self.criterion(logits, yb)
                    if loss_val.dim() > 0:              # p.ej., reduction='none' -> [B]
                        batch_loss = loss_val.mean()
                    else:
                        batch_loss = loss_val

                    bs = yb.size(0)
                    total_loss += batch_loss.item() * bs  # acumular ponderado por tamaño de batch
                    total_samples += bs

                    # predicciones
                    preds = logits.argmax(dim=1)

                    all_preds.append(preds.cpu())
                    all_targets.append(yb.cpu())

            avg_val_loss = total_loss / max(total_samples, 1)
            y_true = torch.cat(all_targets).numpy()
            y_pred = torch.cat(all_preds).numpy()
            f1 = f1_score(y_true, y_pred, average='weighted')

            return avg_val_loss, f1, y_true, y_pred

    def set_params(self, multi_GPU_on=None, **params):
        self.params = params

        # Obtener los parámetros esperados por el constructor de model_class
        #signature = inspect.signature(self.model_class.__init__)
        #valid_keys = set(signature.parameters.keys()) - {'self'}

        # Filtrar los params para incluir solo los esperados
        #filtered_params = {k: v for k, v in params.items() if k in valid_keys}
        #self.model = self.model_class(**filtered_params)
        
        self.model = self.model_class
        if torch.cuda.device_count() > 1 and multi_GPU_on is not None:
            print("Usando", torch.cuda.device_count(), "GPUs")
            self.model = torch.nn.DataParallel(self.model)
        self.optimizer = AdamW(self.model.parameters(), lr=self.params['lr']) 
        self.batch_size = self.params['batch_size']

        # Si el modelo está envuelto en DataParallel, accedemos al .module
        unfreeze_last_layers(self.model, self.params["n_unfreeze"])

    def get_params(self):
        return self.params
              
    def set_criterion(self, y):
          # ----------- Criterion -----------
          if self.sample_weights_loss is not None:
              class_weights = get_sample_weights_loss(y)
              class_weights = torch.tensor(class_weights, dtype=torch.float32).to(self.device)
              self.criterion = nn.CrossEntropyLoss(weight=class_weights)
          else:
              self.criterion = nn.CrossEntropyLoss()

          return self

    def fit_early_stopping(self, train_loader, val_loader, labels):
        #Establecer criterion con sample weights si se especifica
        self.set_criterion(labels)
        self.best_model_state = None
        # ---------- Early stopping (por pérdida) ----------
        patience = 10
        min_delta = 1e-4
        best_val_loss = float('inf')
        epochs_no_improve = 0
        #scheduler
        num_training_steps = len(train_loader) * self.max_epochs
        if self.use_scheduler is not None:
            self.scheduler = get_scheduler(
                "linear", optimizer=self.optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
            )
        #Entrenamiento
        for epoch in range(self.max_epochs):
            self.partial_fit(train_loader)
            avg_val_loss, f1, _, _ = self.predict_and_evaluate(val_loader)
            
            if avg_val_loss + min_delta < best_val_loss:
                best_val_loss = avg_val_loss
                self.best_model_state = self.model.state_dict()
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    break
            print("f1:", f1)
        return f1
    
    def eval_test(self, model_dict, loader):
        all_preds, all_targets = [], []
        model = self.model_class
        model.load_state_dict(model_dict)
        with torch.no_grad():
            for batch in loader:
                # mover batch al device
                batch = {k: v.to(self.device) for k, v in batch.items()}
                xb = {k: v for k, v in batch.items() if k != "labels"}
                yb = batch["labels"]

                outputs = model(**xb)
                logits = outputs.logits

                # predicciones
                preds = logits.argmax(dim=1)

                all_preds.append(preds.cpu())
                all_targets.append(yb.cpu())

        y_true = torch.cat(all_targets).numpy()
        y_pred = torch.cat(all_preds).numpy()
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'f1_score': f1_score(y_true, y_pred, average='macro'),
            'cm': confusion_matrix(y_true, y_pred)
        }
        return metrics
    
    def update_to_best_model(self):
        model = self.model_class
        model.load_state_dict(self.best_model_state)
        self.model = model

#Optuna model
def convert_numpy_to_native(obj):
    if isinstance(obj, dict):
        return {k: convert_numpy_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_to_native(x) for x in obj]
    elif isinstance(obj, np.generic):  # np.float64, np.int64, etc.
        return obj.item()
    else:
        return obj

def get_metrics(y_true, y_pred, verbose = True):
  metrics = {
      'accuracy': accuracy_score(y_true, y_pred),
      'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
      'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
      'f1_score': f1_score(y_true, y_pred, average='weighted'),
      'cm': confusion_matrix(y_true, y_pred)
  }
  if verbose:
    print(metrics)
  return metrics

class optuna_objective_cv:
    def __init__(self, X, y, n_classes, model_name, df_decode, SMOTE_on=None, sample_weights_loss=None, Test_mode = None):
        self.results = {}
        self.X = X
        self.y = y
        self.n_classes = n_classes
        self.sample_weights_loss = sample_weights_loss
        self.max_epochs = 200
        self.best_model_trial = None
        self.Test_mode = Test_mode
        self.df_decode = df_decode
        #BERT models
        self.model_name = model_name
    
    def get_loaders(self, X_train, X_test, y_train, y_test, batch_size):
        train_dataset = TextDataset(list(X_train), y_train, self.tokenizer)
        train_loader = DataLoader(train_dataset, batch_size, shuffle=True)

        test_dataset = TextDataset(list(X_test), y_test, self.tokenizer)
        test_loader = DataLoader(test_dataset, batch_size, shuffle=False)

        return train_loader, test_loader

    def objective(self, trial):
        # ----------- Hiperparámetros a optimizar -----------
        params={
        "lr": trial.suggest_float("lr", 9e-6, 2e-4, log=True),
        "batch_size":12,
        "n_unfreeze":trial.suggest_int("n_unfreeze", 20, 24)
        }
    
        #------------- StratifiedKFold -------------------------------
        F1 = []
        all_metrics = []
        
        """<TEST FUNCTIONS>
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=3)
        for fold, (train_index, test_index) in enumerate(skf.split(self.X, self.y)):
        """ 
        cv_function=CvCustom(self.df_decode)
        for fold, (train_index, test_index) in enumerate(cv_function.split(self.X)):
            #---------------Split data-------------------------------
            X_train, X_test = self.X[train_index], self.X[test_index]
            y_train, y_test = self.y[train_index], self.y[test_index]
            #--------------def model----------------------------------
            model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            pipeline_mlp =  Pytorch_Pipeline(model_class=model, sample_weights_loss = self.sample_weights_loss)
            #Set params
            pipeline_mlp.set_params(**params)
            #Set criterion
            pipeline_mlp.set_criterion(y_train)

            # ------------- Loaders --------------------
            train_loader, test_loader = self.get_loaders(X_train, X_test, y_train, y_test, pipeline_mlp.batch_size)
            # ---------- Early stopping (por loss) ----------
            patience = 10
            min_delta = 1e-4
            best_val_loss = float('inf')
            epochs_no_improve = 0
            best_model_state = None

            for epoch in range(pipeline_mlp.max_epochs):
                pipeline_mlp.partial_fit(train_loader)
                avg_val_loss, f1, y_test, y_pred = pipeline_mlp.predict_and_evaluate(test_loader)
                # ---------- Optuna pruning con F1 ----------
                #Prune only on the first fold
                if fold == 0:
                    trial.report(f1, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
                # ---------- Early stopping (por loss) ----------
                if avg_val_loss + min_delta < best_val_loss:
                    best_val_loss = avg_val_loss
                    best_model_state = pipeline_mlp.model.state_dict()
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                    if epochs_no_improve >= patience:
                        break

            #---------------- Save final result ----------------
            F1.append(f1)
            #-------------Visualization metrics-----------------
            metrics = get_metrics(y_test, y_pred)
            all_metrics.append(metrics)

        #------------ Compute avg among 5 folds ----------------
        mean_F1 = np.mean(F1)

        # ---------- Guarda el modelo del mejor trial según F1 ----------
        try:
            if trial.number == 0 or mean_F1 > trial.study.best_value:
                self.results = {
                    'metrics': self.avg_metrics(all_metrics),
                    'best_params': params,
                    'model_state_dict': best_model_state,
                    'epoch_number': epoch
                }

        except ValueError:
          pass

        #Vaciar memoria
        gc.collect()
        torch.cuda.empty_cache()

        return mean_F1
    
    def avg_metrics(self, all_metrics):
        avg_metrics = {}
        for metric in all_metrics[0].keys():
            values = [metrics[metric] for metrics in all_metrics]

            if metric == 'cm':
                avg_metrics[metric] = np.mean(values, axis=0).astype(int)  # o float si prefieres
            else:
                avg_metrics[metric] = np.mean(values)
        return avg_metrics

    #----------- Método para obtener los resultados -----------
    def get_results(self):
      return self.results
    
#Mlflow functions
def metrics_lang(y, preds, lang_es):
    #Conversión en array
    y = np.array(y)
    preds = np.array(preds)
    lang_es = np.array(lang_es)

    # Seleccionar por máscara booleana
    y_es = y[lang_es] #Data originalmente en español
    preds_es = preds[lang_es]

    y_en = y[~lang_es] #Data originalmente en inglés
    preds_en = preds[~lang_es]
    
    #Computo de métricas
    f1_es = f1_score(y_es, preds_es, average="weighted")
    f1_en = f1_score(y_en, preds_en, average="weighted")
    cm_es = confusion_matrix(y_es, preds_es)
    cm_en = confusion_matrix(y_en, preds_en)

    return f1_es, f1_en, cm_es, cm_en

def eval_model(pipeline_pytorch, test_loader, y_test, lang_es):
  results = {}
  preds = pipeline_pytorch.predict(test_loader)
  cm = confusion_matrix(y_test, preds)
  f1_es, f1_en, cm_es, cm_en = metrics_lang(y_test, preds, lang_es)
  #t_n, f_p, f_n, t_p = cm()
  results = {
      'accuracy': accuracy_score(y_test, preds),
      'precision': precision_score(y_test, preds, zero_division=0),
      'recall': recall_score(y_test, preds, zero_division=0),
      'f1_macro': f1_score(y_test, preds, zero_division=0, average="macro"),
      'cm': cm,
      'f1_es': f1_es,
      'f1_en': f1_en,
      'cm_es': cm_es,
      'cm_en': cm_en
  }
  return results, preds

def safe_log_metric(name, value):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            if np.size(value) == 1:
                value = float(np.array(value).item())
            else:
                raise ValueError("Métrica con más de un valor.")
        else:
            value = float(value)
        mlflow.log_metric(name, value)
    except Exception as e:
        print(f"⚠️ No se pudo loggear {name}: {e}")

def mlflow_ckeckpoint(exp_info, pipeline_pytorch, extra_parms, test_loader, y_test, df_test, mode="server"):
    
    if mode == "server":
        # Set backend store
        mlflow.set_tracking_uri("http://mlflow-server:5000")
        tracking_uri = mlflow.get_tracking_uri()
        print("Current tracking uri: {}".format(tracking_uri)) 
    
    elif mode == "local": 
        # Set backend store
        mlflow.set_tracking_uri(exp_info["tracking_path"])
        tracking_uri = mlflow.get_tracking_uri()
        print("Current tracking uri: {}".format(tracking_uri)) 

        # Verificar si existe experimento, si no crearlo
        experiment = mlflow.get_experiment_by_name(exp_info["exp_name"])

        if experiment is None:
            exp_id = mlflow.create_experiment(
                exp_info["exp_name"],
                artifact_location=exp_info["artifact_path"]
            )
            print(f"Experimento creado con ID: {exp_id}")
        else:
            exp_id = experiment.experiment_id
            print(f"Experimento ya existe con ID: {exp_id}")
    
    else: 
        print("Especificar modo de almacenamiento")
        return 0

    # Define el experimento (lo crea si no existe)
    mlflow.set_experiment(exp_info["exp_name"])
    
    # Obtener commit actual
    repo = git.Repo(search_parent_directories=True)
    commit_hash = repo.head.object.hexsha

    with mlflow.start_run(run_name=exp_info["run_name"]):
        print(f"📝 Registrando modelo en MLflow: {exp_info['run_name']}")

        # Hiperparámetros
        try:
            mlflow.log_params(pipeline_pytorch.get_params())
        except:
            print(f"⚠️ No se pudieron loggear los hiperparámetros para {exp_info['run_name']}")

        #Parámetros adicionales
        for k, v in extra_parms.items():
            mlflow.log_param(k, v)

        # Métricas de test
        results_test, preds = eval_model(pipeline_pytorch, test_loader, y_test, df_test["Español"])
        for k, v in results_test.items():
            if k.startswith("cm"):
                # Guardar confusion matrix (o similar) como artefacto
                # Guardar como CSV temporal
                fname = f"{k}.csv"
                np.savetxt(fname, v, delimiter=",", fmt="%d")

                mlflow.log_artifact(fname, artifact_path="confusion_matrices")

                # Eliminar archivo local si no lo necesitas
                os.remove(fname)

            else:
                # Guardar métrica numérica
                safe_log_metric(f"test_{k}", v)
        
        #Guardar dataframe con predicciones
        fname = f"df_test_preds.csv"
        df_test["preds"]=preds
        df_test["y_test"]=y_test
        df_test.to_csv(fname, index=False, encoding="utf-8-sig")
        mlflow.log_artifact(fname, artifact_path="predictions")
        os.remove(fname)
        
        #Guardar commit de git
        mlflow.log_param("git_commit", commit_hash)

        #Guardar plot de optuna
                
        # Guardar modelo
        mlflow.pytorch.log_model(pipeline_pytorch.model, name = "model")
  

#### Search

In [27]:
#from transformers import logging
#import warnings
#import optuna
#from pipelines.fine_tune_models import optuna_objective_cv, convert_numpy_to_native

warnings.filterwarnings(
    "ignore",
    message="TypedStorage is deprecated"
)
# Desactiva solo los warnings
logging.set_verbosity_error()

#Definir modelo para realizar fine tuning
model_name = "roberta-large"

# ----------- Lanzar la optimización -----------
results_dir = "/tmp/results/finetune/RoBERTa"
os.makedirs(results_dir, exist_ok=True)
#Crear estudio de optuna
study = optuna.create_study(
    direction="maximize",
    study_name="Roberta_ft",
    storage= f"sqlite:///{results_dir}/optuna_1.db",
    load_if_exists=True  # evita sobreescribir si ya existe
)

"""<TEST FUNCTIONS>
X_train, y_train, df_train = gen_dataset(codes_train, df)
X_train=np.array(X_train)[0:50]
# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_train=y_train[0:50]
"""
#Crear función objetivo
X_train = np.array(X_train)
opt_model = optuna_objective_cv(X_train, y_train, df_decode=df_decode, n_classes=2, model_name = model_name,
                                sample_weights_loss=True)
#Optimización
n_trials=20
study.optimize(opt_model.objective, n_trials=n_trials)

# ----------- Mostrar mejores resultados -----------
print("Mejor f1-score:", study.best_value)
print("Mejores hiperparámetros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

#Read results from the best model
results_best_model = opt_model.get_results()

# Aplica la conversión
metrics_native = convert_numpy_to_native(results_best_model['metrics'])

# Imprime con formato limpio
print({'metrics': metrics_native})

[I 2025-09-03 02:04:19,154] Using an existing study with name 'Roberta_ft' instead of creating a new one.


(771,)
{'accuracy': 0.5758754863813229, 'precision': 0.28793774319066145, 'recall': 0.5, 'f1_score': 0.4208867752317817, 'cm': array([[  0, 109],
       [  0, 148]])}
{'accuracy': 0.5758754863813229, 'precision': 0.28793774319066145, 'recall': 0.5, 'f1_score': 0.4208867752317817, 'cm': array([[  0, 109],
       [  0, 148]])}
{'accuracy': 0.42412451361867703, 'precision': 0.21206225680933852, 'recall': 0.5, 'f1_score': 0.2526206119368076, 'cm': array([[109,   0],
       [148,   0]])}


[I 2025-09-03 02:23:57,282] Trial 2 finished with value: 0.364798054133457 and parameters: {'lr': 0.00012662962804111194, 'n_unfreeze': 1}. Best is trial 2 with value: 0.364798054133457.


(771,)
{'accuracy': 0.42412451361867703, 'precision': 0.21206225680933852, 'recall': 0.5, 'f1_score': 0.2526206119368076, 'cm': array([[109,   0],
       [148,   0]])}
{'accuracy': 0.42412451361867703, 'precision': 0.21206225680933852, 'recall': 0.5, 'f1_score': 0.2526206119368076, 'cm': array([[109,   0],
       [148,   0]])}
{'accuracy': 0.5758754863813229, 'precision': 0.28793774319066145, 'recall': 0.5, 'f1_score': 0.4208867752317817, 'cm': array([[  0, 109],
       [  0, 148]])}


[I 2025-09-03 02:52:49,778] Trial 3 finished with value: 0.3087093330351323 and parameters: {'lr': 0.0003665208674441811, 'n_unfreeze': 10}. Best is trial 2 with value: 0.364798054133457.


...

(771,)
{'accuracy': 0.5758754863813229, 'precision': 0.28793774319066145, 'recall': 0.5, 'f1_score': 0.4208867752317817, 'cm': array([[  0, 109],
       [  0, 148]])}
{'accuracy': 0.42412451361867703, 'precision': 0.21206225680933852, 'recall': 0.5, 'f1_score': 0.2526206119368076, 'cm': array([[109,   0],
       [148,   0]])}
{'accuracy': 0.5758754863813229, 'precision': 0.28793774319066145, 'recall': 0.5, 'f1_score': 0.4208867752317817, 'cm': array([[  0, 109],
       [  0, 148]])}


[I 2025-09-03 06:54:53,244] Trial 20 finished with value: 0.364798054133457 and parameters: {'lr': 3.057255304456902e-05, 'n_unfreeze': 7}. Best is trial 5 with value: 0.6065267638068507.


(771,)
{'accuracy': 0.6070038910505836, 'precision': 0.6079043004239855, 'recall': 0.6104326803868088, 'f1_score': 0.6092641045033471, 'cm': array([[69, 40],
       [61, 87]])}
{'accuracy': 0.6498054474708171, 'precision': 0.6402723490102131, 'recall': 0.637924621869576, 'f1_score': 0.6483286180133812, 'cm': array([[ 61,  48],
       [ 42, 106]])}
{'accuracy': 0.5603112840466926, 'precision': 0.5566180935033394, 'recall': 0.5578043639970245, 'f1_score': 0.5625846199191815, 'cm': array([[59, 50],
       [63, 85]])}


[I 2025-09-03 07:14:59,449] Trial 21 finished with value: 0.6067257808119699 and parameters: {'lr': 1.5095930560383809e-05, 'n_unfreeze': 21}. Best is trial 21 with value: 0.6067257808119699.


Mejor f1-score: 0.6067257808119699
Mejores hiperparámetros:
  lr: 1.5095930560383809e-05
  n_unfreeze: 21
{'metrics': {'accuracy': 0.6057068741893644, 'precision': 0.601598247645846, 'recall': 0.6020538887511364, 'f1_score': 0.6067257808119699, 'cm': array([[63, 46],
       [55, 92]])}}


In [5]:
import optuna
results_dir = "/tmp/results/finetune/RoBERTa"
os.makedirs(results_dir, exist_ok=True)
#Crear estudio de optuna
study = optuna.create_study(
    direction="maximize",
    study_name="Roberta_ft",
    storage= f"sqlite:///{results_dir}/optuna_1.db",
    load_if_exists=True  # evita sobreescribir si ya existe
)

[I 2025-09-03 14:21:03,454] Using an existing study with name 'Roberta_ft' instead of creating a new one.


In [6]:
import optuna
from optuna.visualization import plot_param_importances, plot_contour
import matplotlib.pyplot as plt

# ---------- 1. Importancia de Hiperparámetros ----------
fig1 = plot_param_importances(study)
fig1.show()

# ---------- 2. Gráfico de Contorno 2D ----------
# Encuentra los 2 hiperparámetros más importantes
importances = optuna.importance.get_param_importances(study)
top_params = list(importances.keys())[:2]

plot_contour(study, params=["lr", "n_unfreeze"])

In [7]:
for trial in study.trials:
    print(f"Trial {trial.number} | Value: {trial.value} | Params: {trial.params}")

Trial 0 | Value: None | Params: {'lr': 0.00011571681795187999, 'n_unfreeze': 4}
Trial 1 | Value: None | Params: {'lr': 1.3793507947628898e-05, 'n_unfreeze': 22}
Trial 2 | Value: 0.364798054133457 | Params: {'lr': 0.00012662962804111194, 'n_unfreeze': 1}
Trial 3 | Value: 0.3087093330351323 | Params: {'lr': 0.0003665208674441811, 'n_unfreeze': 10}
Trial 4 | Value: 0.3087093330351323 | Params: {'lr': 4.435219051014188e-05, 'n_unfreeze': 3}
Trial 5 | Value: 0.6065267638068507 | Params: {'lr': 1.2695896815189022e-05, 'n_unfreeze': 15}
Trial 6 | Value: 0.364798054133457 | Params: {'lr': 5.48751009369406e-05, 'n_unfreeze': 11}
Trial 7 | Value: 0.3087093330351323 | Params: {'lr': 4.839374555840859e-05, 'n_unfreeze': 20}
Trial 8 | Value: 0.5859366332731438 | Params: {'lr': 1.1848468287726702e-05, 'n_unfreeze': 24}
Trial 9 | Value: 0.3087093330351323 | Params: {'lr': 0.00010040245587096022, 'n_unfreeze': 12}
Trial 10 | Value: 0.2526206119368076 | Params: {'lr': 0.0002099349062543318, 'n_unfreeze

### Retrain

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pipelines.fine_tune_models import Pytorch_Pipeline, mlflow_ckeckpoint
from utils.dataset import CvCustom, TextDataset
from torch.utils.data import DataLoader

#Definir variables
model_name = "roberta-large"
cv_function=CvCustom(df_decode)
X_train=np.array(X_train)

#Sin optuna
params={
    "lr": 1.5095930560383809e-05,
    "batch_size":12,
    "n_unfreeze":21 #24 max
    }

extra_parms={
    "n_trials":20
}
#Reentrenar con mejores hyperparámetros definidos por optuna
#params=study.best_params

#Train
results = []
for nfold, (train_idx, test_idx) in enumerate(cv_function.split(X_train)):
    #Split data
    xt, yt = X_train[train_idx], y_train[train_idx]
    xv, yv = X_train[test_idx], y_train[test_idx]
    #define tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    #Datasets
    train_ds = TextDataset(list(xt), yt, tokenizer)
    val_ds   = TextDataset(list(xv), yv, tokenizer)
    test_ds = TextDataset(list(X_test), y_test, tokenizer)
    #Loaders
    train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=params["batch_size"], shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=params["batch_size"], shuffle=False)
    # ---------- Modelo (capa de clasificación encima de SPECTER) ----------
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    pipeline = Pytorch_Pipeline(model_class=model, use_scheduler=None, max_epochs=200)
    #Train
    pipeline.set_params(**params)
    pipeline.fit_early_stopping(train_loader, val_loader, yt)
    #Get test results
    metrics = pipeline.eval_test(pipeline.best_model_state, test_loader)
    #Save results
    print(metrics)
    results.append(metrics["f1_score"])
    #MLflow
    pipeline.update_to_best_model() #The principal model will be the best model on validation set
    exp_info={
        "exp_name": "RoBERTa_large_finetuning",
        "run_name":f"fold{nfold}"
    }
    mlflow_ckeckpoint(exp_info, pipeline, extra_parms, test_loader, y_test, df_test, mode="server")

mean=np.mean(results)
std=np.std(results)
print("mean:", mean)
print("std:", std)